In [ ]:
# default_exp paper.interpretation.gradshap

# 6.1. GradientShap values

> Calculating features (wavenumbers) importance for the CNN

In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# Python utils
import math
from collections import OrderedDict
from tqdm.auto import tqdm
from pathlib import Path

# mirzai utilities
from mirzai.data.loading import load_kssl
from mirzai.data.selection import (select_y, select_tax_order, select_X)
from mirzai.data.transform import log_transform_y
from mirzai.data.selection import get_y_by_order
#from mirzai.training.cnn import (Model, weights_init)
#from mirzai.data.torch import (DataLoaders, SNV_transform, SpectralDataset)
#from mirzai.training.cnn import Learner


from fastcore.transform import compose



# Data science stack
import numpy as np
from sklearn.model_selection import train_test_split

# Deep Learning stack
import torch
#from torch.nn import MSELoss
#from torch.optim import Adam
#from torch.optim.lr_scheduler import CyclicLR


# Interpretability
import captum
from captum.attr import GradientShap

import warnings
warnings.filterwarnings('ignore')

## Load and transform

In [ ]:
src_dir = '../_data'
fnames = ['spectra-features.npy', 'spectra-wavenumbers.npy', 
          'depth-order.npy', 'target.npy', 
          'tax-order-lu.pkl', 'spectra-id.npy']

X, X_names, depth_order, y, tax_lookup, X_id = load_kssl(src_dir, fnames=fnames)

data = X, y, X_id, depth_order

transforms = [select_y, select_tax_order, select_X, log_transform_y]
X, y, X_id, depth_order = compose(*transforms)(data)

In [ ]:
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Wavenumbers:\n {X_names}')
print(f'depth_order (first 3 rows):\n {depth_order[:3, :]}')
print(f'Taxonomic order lookup:\n {tax_lookup}')

X shape: (40132, 1764)
y shape: (40132,)
Wavenumbers:
 [3999 3997 3995 ...  603  601  599]
depth_order (first 3 rows):
 [[43.  2.]
 [ 0.  0.]
 [ 0.  1.]]
Taxonomic order lookup:
 {'alfisols': 0, 'mollisols': 1, 'inceptisols': 2, 'entisols': 3, 'spodosols': 4, 'undefined': 5, 'ultisols': 6, 'andisols': 7, 'histosols': 8, 'oxisols': 9, 'vertisols': 10, 'aridisols': 11, 'gelisols': 12}


In [ ]:
data = train_test_split(X, y, depth_order, test_size=0.1, random_state=42)
X_train, X_test, y_train, y_test, depth_order_train, depth_order_test = data

#dls = DataLoaders(((X_train, y_train), (X_test, y_test)), transform=SNV_transform())
#training_generator, test_generator = dls.loaders(

## Experiment

In [ ]:
y_by_order, count_by_order, idx_order = get_y_by_order(y_test, depth_order_test[:, 1], tax_lookup)
y_labels = list(np.array([k for k, v in tax_lookup.items()])[idx_order][::-1])

y_labels

['undefined',
 'mollisols',
 'alfisols',
 'inceptisols',
 'ultisols',
 'entisols',
 'aridisols',
 'andisols',
 'vertisols',
 'histosols',
 'spodosols',
 'gelisols',
 'oxisols']

In [ ]:
y_by_order

[array([-0.91477402]),
 array([ 0.37697434, -0.28296551, -0.85009856,  0.50794214, -0.33775447,
        -0.63925686, -0.44563418, -0.44182419,  0.03073298, -0.16506512,
        -0.85996516, -0.90382648, -0.57366916, -0.61011164, -0.66034984,
        -0.76064746, -0.8729234 ,  0.17976511, -0.64118622,  0.57244839,
         0.45413168,  0.37900509,  0.27254281, -0.8562416 , -0.8772492 ,
        -0.48492955,  0.38294878, -0.76414469, -0.49083647, -0.71598436,
        -0.74741612]),
 array([-0.65129355, -0.79950408,  0.09024542, -0.8385191 , -0.09902119,
         0.19003605,  0.20933552,  0.29387708,  0.21128854, -0.25301944,
        -0.36639603, -0.73579912, -0.90063916, -0.7009944 , -0.8315034 ,
         0.38115522, -0.74229383, -0.84996942, -0.75031893, -0.47681926,
        -0.62020642, -0.77294413, -0.85683275, -0.81807528, -0.82756113,
        -0.523265  , -0.72159649,  0.07786874, -0.43283654, -0.84841689,
        -0.76008743, -0.72241898, -0.67880125, -0.84219793, -0.58285594,
     

In [ ]:
def gradShapAgg(X_baseline, X_inspect, model, n_baseline=100, 
                kwargs={'n_samples': 100, 'return_convergence_delta': True}):

    idx_baseline = np.random.randint(X_baseline.shape[0], size=n_baseline)
    X_baseline = X_train[idx_baseline, :]

    gs = GradientShap(model, multiply_by_inputs=True)
    shaps = []
    for i in tqdm(range(len(X_inspect))):
        gs_attr_test, delta = gs.attribute(X_to_torch(X_inspect[[i],:]), baselines=X_to_torch(X_baseline), 
                                           **kwargs)
        shaps.append(gs_attr_test.cpu().detach().numpy().ravel())

    return np.array(shaps)

['undefined',
 'mollisols',
 'alfisols',
 'inceptisols',
 'ultisols',
 'entisols',
 'aridisols',
 'andisols',
 'vertisols',
 'histosols',
 'spodosols',
 'gelisols',
 'oxisols']

In [ ]:
y_by_order, count_by_order, idx_order = get_y_by_order(y_test, depth_order_test[:, 1], tax_lookup)
y_labels = list(np.array([k for k, v in tax_lookup.items()])[idx_order][::-1])

shap_groups = []
X_shap_groups = []
for order in tqdm(['all'] + y_labels):
    if order == 'all': 
        X_inspect = X_test
    else:
        idx = tax_lookup[order]
        mask = depth_order_test[:, 1] == idx
        X_inspect = X_test[mask, :]
    
    X_shap_groups.append(X_inspect)
    shaps = gradShapAgg(X_train, X_inspect, model)
    shap_groups.append(OrderedDict({'shaps': np.mean(shaps, axis=0), 'order': order}))

In [ ]:
X_shap_groups, shap_groups = pickle.load(open(os.path.join(MONITORING_PATH, 'shap_groups.pickle'), "rb"))

In [ ]:
tax_pretty_lut = {'all': 'all', 'undefined': 'undefined', 'alfisols': 'alfi.', 'andisols': 'andi.',
                  'aridisols': 'aridi.', 'entisols': 'enti.', 'gelisols': 'geli.', 'histosols': 'histo.',
                  'inceptisols': 'incepti.', 'mollisols': 'molli.', 'oxisols': 'oxi.', 'spodosols': 'spodo.',
                  'ultisols': 'ulti.', 'vertisols': 'verti.'}

In [ ]:
def prettify_label(label, tax_lut):
    return tax_lut[label].capitalize(

In [ ]:
def plot_shaps_by_group(attr_values, X, X_names, tax_pretty, diverging=False,
                        annotate=True, figsize=(16*centimeter,6*centimeter), dpi=600):
    # Styles
    p = plt.rcParams
    p["font.size"] = 8

    p["axes.linewidth"] = 1
    p["axes.facecolor"] = "white"
    p["axes.ymargin"] = 0.11
    p["axes.spines.bottom"] = False
    p["axes.spines.left"] = False
    p["axes.spines.right"] = False
    p["axes.spines.top"] = False
    p['axes.linewidth'] = 0.5 
    # p['axes.labelsize'] = 'small'

    p["axes.grid"] = False
    p["grid.color"] = "black"
    p["grid.linewidth"] = 0.2
    p['grid.linestyle'] = '-'

    p["xtick.labelsize"] = 6
    p["xtick.bottom"] = True
    p["xtick.top"] = False
    p["xtick.direction"] = "in"
    p["xtick.major.size"] = 3
    p["xtick.major.width"] = 0.5
    p["xtick.minor.size"] = 1
    p["xtick.minor.width"] = 0.25
    p["xtick.minor.visible"] = True

    p["ytick.left"] = False
    p["ytick.right"] = False 
    p["ytick.labelleft"] = False
    p["ytick.labelright"] = False
    p["ytick.direction"] = "in"
    # p["ytick.major.size"] = 5
    p["ytick.major.size"] = 3
    p["ytick.major.width"] = 0.5
    p["ytick.minor.size"] = 1
    p["ytick.minor.width"] = 0.25
    p["ytick.minor.visible"] = False

    p['boxplot.medianprops.color'] =  'white'
    p['boxplot.medianprops.linewidth'] = 1.0

    # Layout 
    # fig = plt.figure(figsize=figsize, dpi=dpi)
    fig, axes = plt.subplots(ncols=1, nrows=len(attr_values),
                             sharey=False, figsize=figsize, dpi=dpi) 
    
    # Calculate color scale adapted to grid resolution
    for i, ax in enumerate(axes.flat):
        shap, label = attr_values[i].values() 
        if not diverging:
            shap = np.absolute(shap)
        ax.set_xlim(np.max(X_names), np.min(X_names))
        # title = 'All' if label == 'all' else label.capitalize()

        # title = shorten_label(title, max_len=5) 
        title = prettify_label(label, tax_pretty_lut) 
        # ax.set_title(f'{title}', loc='left')

        if diverging:
            mask_pos = shap > 0
            attr_pos = np.copy(shap)
            attr_neg = np.copy(shap)
            attr_pos[~mask_pos] = 0   
            attr_neg[mask_pos] = 0   
            ax.bar(X_names, attr_pos, width=2, color='#0571b0', label='GradientShap > 0')
            ax.bar(X_names, attr_neg, width=2, color='#ca0020', label='GradientShap < 0')
            # ax.set_ylabel('IG') 
        else:
            ax.bar(X_names, shap, width=2, color='black')
        
        # ax.yaxis.set_major_formatter(FormatStrFormatter('%.0e'))
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f')) 
        ax.set_ylabel(f'{title}')
        ax.get_yaxis().set_ticks([])

        ax_twin = ax.twinx()
        ax_twin.plot(X_names, np.mean(X[i], axis=0), c='#555', 
                     alpha=1, lw=0.5, ls='--', zorder=-1, 
                     label='Mean spectrum (Absorbance)') 
        ax_twin.get_yaxis().set_ticks([])

        ax.xaxis.set_major_locator(ticker.MaxNLocator(20))
        ax.xaxis.set_minor_locator(ticker.MaxNLocator(80))

    handles_all = []
    labels_all = []
    for ax in [axes[0], ax_twin]:
        handles, labs = ax.get_legend_handles_labels()
        labels_all += labs
        handles_all += handles
    
    # fig.text(1, 0.5, 'Absorbance →', ha='center', va='center', rotation='vertical')
    fig.legend(handles_all, labels_all, 
               frameon=False, ncol=5, loc='upper center',  borderaxespad=0.1) 
    
   # Ornaments
    #ax.set_ylabel('$\overline{| Attribution |}$ →', loc='top')
    axes.flat[-1].set_xlabel('Wavenumber ($cm^{-1}$) →', loc='right')
    # fig.text(0, 0.5, 'GradShap mean attribution', ha='center', va='center', rotation='vertical') 
        # ax.grid(True, "minor", axis='x', color="0.65", linewidth=0.2, zorder=-20)
        # ax.grid(True, "major", axis='x', color="0.65", linewidth=0.2, zorder=-10) 
 
    plt.tight_layout()
    plt.savefig(os.path.join(IMG_PATH, 'gradshap-order.png'), dpi=600, transparent=True, format='png')

In [ ]:
plot_shaps_by_group(shap_groups, X_shap_groups, X_names, tax_pretty_lut,
                    figsize=(16*centimeter, 20*centimeter), diverging=True)

### Setup

In [ ]:
# Is a GPU available?
use_cuda = torch.cuda.is_available()
device = torch.device('cuda:0' if use_cuda else 'cpu')
print(f'Runtime is: {device}')

Runtime is: cpu


In [ ]:
seeds = range(20)
criterion = MSELoss() # Mean Squared Error loss
min_epoch, max_epoch = 10, 210
base_lr, max_lr = 3e-5, 1e-3

epoch_inc = 10
step_size_up = 5

if device.type == 'cpu':
    min_epoch, max_epoch = 1, 3
    epoch_inc = 1
    seeds = range(2)
    
tax_of_interest = tax_lookup.copy()
tax_of_interest['all'] = tax_of_interest.pop('oxisols') # Too few Oxisols for now in the KSSL db    

mses_by_order = np.zeros((len(seeds),
                          len(range(min_epoch, max_epoch, epoch_inc)),
                          len(tax_of_interest),
                          2))

### Run

In [ ]:
for seed in tqdm(seeds):
    print("************")
    print('Seed: ', seed)
    
    # 1. Initialize a "fresh" new CNN
    model = Model(X.shape[1], out_channel=16).to(device)
    opt = Adam(model.parameters(), lr=1e-4)
    model = model.apply(weights_init)
    scheduler = CyclicLR(opt, base_lr=base_lr, max_lr=max_lr,
                         step_size_up=step_size_up, mode='triangular',
                         cycle_momentum=False)

    # 2. Starting from min_epoch, train additional 10 epochs then stop when reaching max_epoch 
    for epoch_idx, epoch in enumerate(range(min_epoch, max_epoch, epoch_inc)):
        print('Epoch: ', epoch)
        data = train_test_split(X, y, depth_order, test_size=0.1, random_state=seed)
        X_train, X_test, y_train, y_test, depth_order_train, depth_order_test = data
        dls = DataLoaders(((X_train, y_train), (X_test, y_test)), transform=SNV_transform())
        training_generator, test_generator = dls.loaders()
            
        # Train addition `epoch_inc` epochs
        learner = Learner(model, criterion, opt, n_epochs=epoch_inc, 
                          scheduler=scheduler, verbose=True)
        model, losses = learner.fit(training_generator, test_generator) 
               
        # 3. Evaluate on train & test sets on each Soil Taxonomy orders of interest
        for tax_idx in range(len(tax_of_interest)):
            if tax_idx == 9:
                mask_train = np.ones(len(depth_order_train), dtype=bool)
                mask_test = np.ones(len(depth_order_test), dtype=bool)
            else:
                mask_train = depth_order_train[:, 1] == tax_idx
                mask_test = depth_order_test[:, 1] == tax_idx
            if np.sum(mask_test) != 0:
                training_set_order = SpectralDataset(X_train[mask_train, :], y_train[mask_train], transform=SNV_transform())
                test_set_order = SpectralDataset(X_test[mask_test, :], y_test[mask_test], transform=SNV_transform())

                #y_hat_train, y_true_train = predict(model, training_set_order)
                y_hat_train, y_true_train = learner.predict(training_set_order)
                mse_train = eval_reg(y_true_train, y_hat_train, is_log=True)['mse']

                #y_hat_test, y_true_test = predict(model, test_set_order)
                y_hat_test, y_true_test = learner.predict(test_set_order)
                mse_test = eval_reg(y_true_test, y_hat_test, is_log=True)['mse']

                mses_by_order[seed, epoch_idx, tax_idx, 0] = mse_train
                mses_by_order[seed, epoch_idx, tax_idx, 1] = mse_test
            else:
                deltas_mse[seed, epoch_idx, tax_idx] = np.nan


  0%|          | 0/2 [00:00<?, ?it/s]

************
Seed:  0
Epoch:  1
Runtime is: cpu


  0%|          | 0/1 [00:00<?, ?it/s]

End of Epoch 1
 Training loss: 0.125955143588718
 Validation loss: 0.10428656276965899


TypeError: 'int' object is not callable

## Save

In [ ]:
MONITORING_PATH = Path('nameofyourfolder')
with open(MONITORING_PATH/'mses_by_order_90_10.pickle'), 'wb') as f: 
    pickle.dump(mses_by_order, f)